In [ ]:
%pip install --upgrade openai pydantic mermaid-py

# Web Research + Structured Report Agent

In this notebook, you will build an AI agent that uses OpenAI's `web_search_preview` tool to research topics on the live web and produce structured, citation-backed reports using the Responses API.

## Architecture Overview

Your **Web Research + Structured Report Agent** follows a three-phase pattern:

1. **Query Generation** – The model decomposes a broad research topic into multiple targeted search queries.
2. **Web Search Execution** – Each query is executed using the `web_search_preview` tool, which gives the model access to live web data.
3. **Synthesis & Structuring** – All search results are synthesized into a structured JSON report with citations.

The `web_search_preview` tool is a built-in tool in the OpenAI Responses API. When provided, the model can autonomously decide to search the web and incorporate real-time information into its response.

In [ ]:
import mermaid as md

md.Mermaid("""
flowchart TD
    A[Research Topic] --> B[Generate Targeted Search Queries]
    B --> C1[Search #1]
    B --> C2[Search #2]
    B --> C3[Search #3]
    C1 --> D[Synthesize into Structured Report]
    C2 --> D
    C3 --> D
""")

In [ ]:
import os
import json
import re
from getpass import getpass
from pydantic import BaseModel, Field
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-5-mini"


def parse_json_response(text):
    """Parse JSON from a model response, stripping markdown fences if present."""
    cleaned = re.sub(r"^```(?:json)?\s*", "", text.strip())
    cleaned = re.sub(r"\s*```$", "", cleaned)
    return json.loads(cleaned)

## Part 1: Basic Web Search

The simplest way to use web search with the Responses API is to include `web_search_preview` in the `tools` list. The model will automatically decide when to search and will incorporate web results into its response.

Let's try it out with a question that requires up-to-date information.

In [ ]:
# Define the web search tool
tools = [{"type": "web_search_preview"}]

# Make a request that requires up-to-date information
response = client.responses.create(
    model=MODEL,
    tools=tools,
    input="What are the latest developments in quantum computing?"
)

print(response.output_text)

### Inspecting Web Search Results

The `response.output` array contains multiple items when web search is used. Let's see what the model found behind the scenes:

- **`web_search_call`** items show which searches the model performed.
- **Message output** items contain the final text with source annotations (URLs and titles).

In [ ]:
# Inspect the response output to see search calls and source annotations
print("Response output items:")
print(f"Total items: {len(response.output)}\n")

for item in response.output:
    if item.type == "web_search_call":
        print(f"[Web Search] ID: {item.id}")
        print(f"  Status: {item.status}")
    elif item.type == "message":
        # This is the message output containing the final response
        print(f"\n[Message] Role: {item.role}")
        for content in item.content:
            if hasattr(content, "annotations") and content.annotations:
                print(f"\nSources cited ({len(content.annotations)} annotations):")
                for annotation in content.annotations:
                    print(f"  - {annotation.title}")
                    print(f"    URL: {annotation.url}")

## Part 2: Web Search with Custom Configuration

The `web_search_preview` tool supports additional configuration options:

- **`user_location`** – Provide an approximate location to get localized results (country, city, region).
- **`search_context_size`** – Control how much search context the model receives. Options are `"low"`, `"medium"` (default), or `"high"`. Higher values use more tokens but provide richer context.

Let's configure both of these and see the difference.

In [ ]:
# Configure web search with location and context size
tools_configured = [{
    "type": "web_search_preview",
    "user_location": {
        "type": "approximate",
        "country": "US",
        "city": "San Francisco",
        "region": "California"
    },
    "search_context_size": "medium"
}]

response_local = client.responses.create(
    model=MODEL,
    tools=tools_configured,
    input="What are the top AI research labs and their recent breakthroughs?"
)

print(response_local.output_text)

## Part 3: Multi-Step Research Agent

A single web search is useful, but for comprehensive research you need a **multi-step agent** that:

1. **Decomposes** a broad topic into specific, targeted search queries.
2. **Executes** each search independently to gather diverse information.
3. **Synthesizes** all findings into a coherent, structured report.
4. **Includes citations** from all sources for verifiability.

This pattern is powerful because it mirrors how a human researcher would approach a topic: break it down, gather information from multiple angles, then synthesize everything into a single narrative.

### Define the Research Function

The `research_topic()` function orchestrates the entire research pipeline. It takes a topic and the number of searches to perform, then returns a structured report as a Python dictionary.

In [ ]:
def research_topic(topic, num_searches=3):
    """
    Multi-step research agent that performs targeted web searches
    and produces a structured report with citations.

    Args:
        topic: The research topic to investigate.
        num_searches: Number of targeted searches to perform (default: 3).

    Returns:
        A dictionary containing the structured research report.
    """
    # Step 1: Generate targeted search queries
    print(f"Step 1: Generating {num_searches} targeted search queries...")
    print("-" * 50)

    query_response = client.responses.create(
        model=MODEL,
        instructions=(
            f"Generate {num_searches} specific search queries to thoroughly "
            f"research this topic. Each query should target a different aspect "
            f"of the topic. Return them as a JSON array of strings. "
            f"Only return the JSON array, nothing else."
        ),
        input=f"Research topic: {topic}"
    )

    queries = parse_json_response(query_response.output_text)
    print(f"Generated {len(queries)} search queries:")
    for i, q in enumerate(queries, 1):
        print(f"  {i}. {q}")

    # Step 2: Execute each search and collect results
    print(f"\nStep 2: Executing web searches...")
    print("-" * 50)

    tools = [{"type": "web_search_preview"}]
    search_results = []

    for i, query in enumerate(queries, 1):
        print(f"\n  Searching [{i}/{len(queries)}]: {query}")

        response = client.responses.create(
            model=MODEL,
            tools=tools,
            instructions=(
                "Search for this information and provide a detailed summary "
                "with key facts, statistics, and findings. "
                "Include source URLs where possible."
            ),
            input=query
        )

        search_results.append({
            "query": query,
            "findings": response.output_text
        })
        print(f"  Done - collected {len(response.output_text)} chars of findings.")

    # Step 3: Synthesize into a structured report
    print(f"\nStep 3: Synthesizing findings into structured report...")
    print("-" * 50)

    synthesis_prompt = f"""Based on the following research findings, create a comprehensive structured report.

Topic: {topic}

Research Findings:
{json.dumps(search_results, indent=2)}

Return a JSON object with this exact structure:
{{
    "title": "Report title",
    "executive_summary": "2-3 sentence summary of the overall findings",
    "key_findings": ["finding 1", "finding 2", ...],
    "detailed_sections": [
        {{"heading": "Section title", "content": "Section content with facts and data"}},
        ...
    ],
    "sources": ["source 1 URL or description", ...],
    "confidence_notes": "Any caveats or areas where information was limited"
}}

Only return valid JSON. Do not wrap it in markdown code fences."""

    report_response = client.responses.create(
        model=MODEL,
        instructions=(
            "You are a research analyst. Synthesize the provided research "
            "findings into a well-structured report. Be factual, cite specific "
            "data points, and note any contradictions between sources."
        ),
        input=synthesis_prompt
    )

    report = parse_json_response(report_response.output_text)
    print("Report generated successfully!")
    return report

### Run the Research Agent

Now let's put your agent to work on a real research topic. The agent will generate queries, search the web, and produce a structured report. Watch the progress as it works through each step.

In [ ]:
topic = (
    "The current state of large language model efficiency techniques "
    "(quantization, distillation, pruning) in 2025"
)

report = research_topic(topic, num_searches=3)

In [ ]:
# Pretty-print the research report
def display_report(report):
    """Display a structured research report in a readable format.

    Accepts both dict and Pydantic model inputs.
    """
    # Convert Pydantic model to dict if needed
    if hasattr(report, 'model_dump'):
        report = report.model_dump()

    print(f"\n{'=' * 60}")
    print(f"RESEARCH REPORT: {report['title']}")
    print(f"{'=' * 60}")

    print(f"\nExecutive Summary:")
    print(f"  {report['executive_summary']}")

    print(f"\nKey Findings:")
    for i, finding in enumerate(report["key_findings"], 1):
        print(f"  {i}. {finding}")

    print(f"\nDetailed Sections:")
    for section in report["detailed_sections"]:
        print(f"\n  --- {section['heading']} ---")
        print(f"  {section['content']}")

    print(f"\nSources:")
    for i, source in enumerate(report["sources"], 1):
        print(f"  [{i}] {source}")

    if report.get("confidence_notes"):
        print(f"\nConfidence Notes:")
        print(f"  {report['confidence_notes']}")

display_report(report)

## Part 4: Structured Output with JSON Schema

In the research function above, you asked the model to return JSON and then parsed it manually. This works, but there is a more robust approach: using the Responses API's **structured output** feature with `text.format`.

By providing a JSON schema (derived from a Pydantic model), you guarantee the model's output conforms to the exact structure you need. This eliminates parsing errors and ensures every field is present. Let's define our schema.

In [ ]:
from pydantic import BaseModel, Field


class DetailedSection(BaseModel):
    """A section of the research report."""
    heading: str = Field(description="Section title")
    content: str = Field(description="Section content with facts and data")


class ResearchReport(BaseModel):
    """Structured research report with citations."""
    title: str = Field(description="Report title")
    executive_summary: str = Field(description="2-3 sentence summary of overall findings")
    key_findings: list[str] = Field(description="List of key findings")
    detailed_sections: list[DetailedSection] = Field(description="Detailed report sections")
    sources: list[str] = Field(description="Source URLs or descriptions")
    confidence_notes: str = Field(description="Caveats or areas where information was limited")


# Generate the schema from the Pydantic model
report_schema = {
    "type": "json_schema",
    "name": "research_report",
    "strict": True,
    "schema": ResearchReport.model_json_schema()
}

print("Schema defined from Pydantic model. Fields:")
for field_name in ResearchReport.model_fields:
    print(f"  - {field_name}")

In [ ]:
def research_topic_structured(topic, num_searches=3):
    """
    Enhanced research agent that uses Pydantic models and JSON Schema
    structured output to guarantee a consistent report format.

    Args:
        topic: The research topic to investigate.
        num_searches: Number of targeted searches to perform (default: 3).

    Returns:
        A ResearchReport Pydantic model containing the structured report.
    """
    # Step 1: Generate targeted search queries
    print(f"Step 1: Generating {num_searches} targeted search queries...")
    print("-" * 50)

    query_response = client.responses.create(
        model=MODEL,
        instructions=(
            f"Generate {num_searches} specific search queries to thoroughly "
            f"research this topic. Each query should target a different aspect "
            f"of the topic. Return them as a JSON array of strings. "
            f"Only return the JSON array, nothing else."
        ),
        input=f"Research topic: {topic}"
    )

    queries = parse_json_response(query_response.output_text)
    print(f"Generated {len(queries)} search queries:")
    for i, q in enumerate(queries, 1):
        print(f"  {i}. {q}")

    # Step 2: Execute each search and collect results
    print(f"\nStep 2: Executing web searches...")
    print("-" * 50)

    tools = [{"type": "web_search_preview"}]
    search_results = []

    for i, query in enumerate(queries, 1):
        print(f"\n  Searching [{i}/{len(queries)}]: {query}")

        response = client.responses.create(
            model=MODEL,
            tools=tools,
            instructions=(
                "Search for this information and provide a detailed summary "
                "with key facts, statistics, and findings. "
                "Include source URLs where possible."
            ),
            input=query
        )

        search_results.append({
            "query": query,
            "findings": response.output_text
        })
        print(f"  Done - collected {len(response.output_text)} chars of findings.")

    # Step 3: Synthesize with Pydantic structured output
    print(f"\nStep 3: Synthesizing with Pydantic structured output...")
    print("-" * 50)

    synthesis_prompt = f"""Based on the following research findings, create a comprehensive structured report.

Topic: {topic}

Research Findings:
{json.dumps(search_results, indent=2)}

Provide a thorough report with an executive summary, key findings,
detailed sections covering each aspect, source citations, and any
confidence notes about information quality or gaps."""

    report_response = client.responses.create(
        model=MODEL,
        instructions=(
            "You are a research analyst. Synthesize the provided research "
            "findings into a well-structured report. Be factual, cite specific "
            "data points, and note any contradictions between sources."
        ),
        input=synthesis_prompt,
        text={"format": report_schema}
    )

    report = ResearchReport.model_validate_json(report_response.output_text)
    print("Report generated and validated with Pydantic!")
    return report

### Run Another Research Topic

Let's test the structured-output version of your agent with a different topic. This demonstrates how versatile the pattern is; you can swap in any topic and get a consistently formatted report back.

In [ ]:
topic_2 = "Impact of AI agents on software development workflows"

report_2 = research_topic_structured(topic_2, num_searches=3)

In [ ]:
display_report(report_2)

In [ ]:
# The structured output returns a validated Pydantic model
print("Pydantic model validation check:")
print(f"  Type: {type(report_2).__name__}")
print()

for field_name, field_info in ResearchReport.model_fields.items():
    value = getattr(report_2, field_name)
    field_type = type(value).__name__
    print(f"  [PASS] {field_name}: {field_type}")

print(f"\nReport contains {len(report_2.key_findings)} key findings")
print(f"Report contains {len(report_2.detailed_sections)} detailed sections")
print(f"Report cites {len(report_2.sources)} sources")

## Your Turn: Build a Fact-Checker Agent

Time to practice! You will build an agent that takes a claim and uses `web_search_preview` to verify whether it is true, false, or partially true.

## Exercise 1: Build a Fact-Checker Agent

Your goal is to build an agent that takes a claim (for example,
"Python is the most popular programming language in 2026") and uses the
`web_search_preview` tool to verify whether it is true, false, or
partially true.

**What your agent should do:**

1. Accept a claim as a string.
2. Use `web_search_preview` to search the web for evidence.
3. Return a `FactCheckResult` Pydantic model containing:
   - A **verdict** (`true`, `false`, or `partially_true`)
   - A **confidence** score between 0.0 and 1.0
   - A list of **evidence** strings supporting the verdict
   - A list of **source** URLs

**Hints:**

- You can accomplish this in a single `client.responses.create()` call by
  combining `web_search_preview` with structured output (`text.format`).
- Use `instructions` to tell the model it is a fact-checker and should
  search the web before making a judgement.
- Derive the JSON schema from the Pydantic model with
  `FactCheckResult.model_json_schema()`.
- Parse the response with `FactCheckResult.model_validate_json(response.output_text)`.

In [ ]:
class FactCheckResult(BaseModel):
    verdict: str = Field(description="One of: true, false, partially_true")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    evidence: list[str] = Field(description="Key evidence supporting the verdict")
    sources: list[str] = Field(description="Source URLs")

### Now it is your turn to build the `fact_check` function!

Fill in the function body below. You need to:

1. Build the `text.format` schema dict from `FactCheckResult`.
2. Call `client.responses.create()` with `web_search_preview` as a tool
   and structured output via `text={"format": ...}`.
3. Parse the response into a `FactCheckResult` and return it.

In [ ]:
def fact_check(claim: str) -> FactCheckResult:
    """Verify a claim using web search and return a structured verdict."""

    # YOUR CODE HERE
    # Step 1: Build the JSON schema for structured output
    # fact_check_schema = {
    #     "type": "json_schema",
    #     "name": "fact_check_result",
    #     "strict": True,
    #     "schema": ...
    # }

    # Step 2: Call client.responses.create() with:
    #   - model=MODEL
    #   - tools=[{"type": "web_search_preview"}]
    #   - instructions telling the model to act as a fact-checker
    #   - input=claim
    #   - text={"format": fact_check_schema}

    # Step 3: Parse and return
    # return FactCheckResult.model_validate_json(response.output_text)

    pass  # Remove this line when you add your implementation

### Test your fact-checker

Run the cell below to verify your implementation. If everything works,
you should see a verdict, confidence score, evidence, and sources.

In [ ]:
result = fact_check("Python is the most popular programming language in 2026")

print(f"Verdict: {result.verdict}")
print(f"Confidence: {result.confidence}")
print()
print("Evidence:")
for e in result.evidence:
    print(f"  - {e}")
print()
print("Sources:")
for s in result.sources:
    print(f"  - {s}")

## Summary

Great work! In this notebook, you built a **Web Research + Structured Report Agent** using the OpenAI Responses API. Here are the key takeaways:

- **The `web_search_preview` tool** gives models access to live web data, enabling real-time research without any external API setup.
- **Multi-step research agents** decompose broad topics into targeted search queries, producing more comprehensive and diverse results than a single query.
- **Structured outputs** (via `text.format` with a JSON schema) guarantee the report conforms to a consistent format, eliminating parsing errors and ensuring every required field is present.
- **Citations and source tracking** are essential for research quality. The web search tool automatically provides source annotations that you can extract and include in reports.
- **The core pattern** is: *generate queries* > *search the web* > *synthesize findings* > *structure the output*. This pattern is adaptable to many research and analysis use cases.

Try swapping in your own topics and adjusting the number of searches to see how the depth of the report changes!